# Workflow end-to-end — test offline

Questo notebook usa i **13 pack del manifest reale** (inclusi i tre mini-pack di Lorenzo) con dati sintetici validi restituiti da un fake client YGOPRODeck. Tutte le scritture avvengono in `TemporaryDirectory`; non viene modificata l'installazione reale.

**Prima di eseguirlo**, in `workflow.py` sposta `save_id_registry(...)` dopo la scrittura e validazione finale della build.


In [ ]:
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import patch
import json
import shutil
import yaml

import ygomaster_packbuilder.workflow as workflow_module
from ygomaster_packbuilder.workflow import WorkflowError, resolve_workflow_paths, build_project, build_and_install_project
from ygomaster_packbuilder.installer import uninstall_ygomaster_build
from ygomaster_packbuilder.installer.state import load_install_state
from ygomaster_packbuilder.registry import load_id_registry

def find_root():
    cwd = Path.cwd().resolve()
    for folder in (cwd, *cwd.parents):
        if (folder / "pyproject.toml").is_file() and (folder / "config" / "project.yaml").is_file():
            return folder
    raise FileNotFoundError("Apri il notebook all'interno della repository.")

ROOT = find_root()
MANIFEST = yaml.safe_load((ROOT / "config" / "project.yaml").read_text(encoding="utf-8"))
assert len(MANIFEST["players"]) == 3
assert len(MANIFEST["shared_packs"]) == 1
assert all(len(p["mini_packs"]) == 3 for p in MANIFEST["players"])
assert all(p["source"]["external_id"] != "XXXXX" for p in MANIFEST["players"] for p in p["mini_packs"])

class FakeYgoProDeckClient:
    def __init__(self, packs):
        self.packs, self.calls = packs, []
    def fetch_custom_pack_raw(self, external_id):
        external_id = str(external_id)
        self.calls.append(external_id)
        return deepcopy(self.packs[external_id])

def make_sources(manifest):
    sources, mapping = {}, {}
    next_ydk, next_md = 100_000, 1_000
    def cards(count, rarity):
        nonlocal next_ydk, next_md
        result = []
        for _ in range(count):
            next_ydk += 1
            next_md += 1
            result.append({"id": next_ydk, "name": f"Card {next_ydk}",
                           "type": "Effect Monster", "card_rarity": rarity})
            mapping[next_ydk] = next_md
        return result

    def put(pack, selected):
        ext_id = str(pack["source"]["external_id"])
        assert ext_id not in sources
        sources[ext_id] = [
            {**card, "pack_name": pack["key"], "pack_description": "Synthetic",
             "username": "Offline", "img_url": "fixture.png"}
            for card in selected
        ]

    for player in manifest["players"]:
        main = player["main_pack"]
        groups = {
            "N": cards(48, "Common") + cards(2, "Secret Rare"),
            "R": cards(25, "Rare"),
            "SR": cards(15, "Super Rare"),
            "UR": cards(10, "Ultra Rare"),
        }
        put(main, sum(groups.values(), []))
        for i, mini in enumerate(player["mini_packs"]):
            selection = (
                groups["N"][i*10:(i+1)*10] + groups["R"][i*5:(i+1)*5]
                + groups["SR"][i*3:(i+1)*3] + groups["UR"][i*2:(i+1)*2]
            )
            assert len(selection) == 20
            put(mini, selection)

    staples = (
        cards(100, "Common") + cards(60, "Rare")
        + cards(20, "Super Rare") + cards(14, "Ultra Rare")
        + cards(6, "Secret Rare")
    )
    put(manifest["shared_packs"][0], staples)
    return sources, mapping

SOURCES, MAPPING = make_sources(MANIFEST)
assert len(SOURCES) == 13 and len(MAPPING) == 500
assert {"24390", "24393", "24395"}.issubset(SOURCES)
print("Fixture:", len(SOURCES), "pack,", len(MAPPING), "carte")


## Workspace temporanea

In [ ]:
SHOP_TEXT = '''{
    "PerPackRarities": false,
    // Questo commento deve sopravvivere
    "PackShop": {}
}
'''

def prepare_workspace(base, mapping=MAPPING):
    base = Path(base)
    project_root = base / "PackBuilder"
    ygomaster_dir = base / "YgoMaster"
    (project_root / "config").mkdir(parents=True)
    (ygomaster_dir / "Data").mkdir(parents=True)
    for name in ("project.yaml", "policies.yaml", "rarity_mapping.yaml"):
        shutil.copy2(ROOT / "config" / name, project_root / "config" / name)
    (ygomaster_dir / "Data" / "YdkIds.txt").write_text(
        "".join(f"{ydk} {md}\n" for ydk, md in sorted(mapping.items())),
        encoding="utf-8",
    )
    (ygomaster_dir / "Data" / "Shop.json").write_bytes(SHOP_TEXT.encode("utf-8"))
    return project_root, ygomaster_dir

print("Workspace pronta")


## 1. Build completa su 13 pack reali del manifest (fake HTTP)

In [ ]:
with TemporaryDirectory() as tmp:
    project_root, ygomaster_dir = prepare_workspace(tmp)
    paths = resolve_workflow_paths(project_root, ygomaster_dir)
    assert paths.ydk_mapping == ygomaster_dir / "Data" / "YdkIds.txt"
    assert paths.build_dir == project_root / "build"

    client = FakeYgoProDeckClient(SOURCES)
    result = build_project(project_root, ygomaster_dir, ygoprodeck_client=client)
    assert len(client.calls) == 13
    assert len(result.project.players) == 3
    assert len(result.project.packs) == 13
    assert len(result.pack_ids) == len(set(result.pack_ids.values())) == 13
    assert all(9_000_000 <= i <= 9_999_999 for i in result.pack_ids.values())
    assert result.artifacts.shop.is_file()
    assert result.artifacts.shop_pack_odds.is_file()
    assert result.artifacts.shop_settings_patch.is_file()

    shop = json.loads(result.artifacts.shop.read_text(encoding="utf-8"))
    odds = json.loads(result.artifacts.shop_pack_odds.read_text(encoding="utf-8"))
    assert len(shop["PackShop"]) == len(odds) == 13
    assert set(shop["PackShop"]) == {str(10_000_000 + i) for i in result.pack_ids.values()}
    assert load_id_registry(paths.id_registry).pack_ids == result.pack_ids
    assert list(project_root.glob(".ygomaster-packbuilder-*")) == []
    print("Build completa OK", len(result.project.packs), "pack")


## 2. Ripetizione: stable IDs e output deterministico

In [ ]:
with TemporaryDirectory() as tmp:
    project_root, ygomaster_dir = prepare_workspace(tmp)
    a = build_project(project_root, ygomaster_dir, ygoprodeck_client=FakeYgoProDeckClient(SOURCES))
    old_shop = a.artifacts.shop.read_bytes()
    old_odds = a.artifacts.shop_pack_odds.read_bytes()
    old_registry = (project_root / "state" / "ygomaster_ids.json").read_bytes()

    b = build_project(project_root, ygomaster_dir, ygoprodeck_client=FakeYgoProDeckClient(SOURCES))
    assert b.pack_ids == a.pack_ids
    assert b.artifacts.shop.read_bytes() == old_shop
    assert b.artifacts.shop_pack_odds.read_bytes() == old_odds
    assert (project_root / "state" / "ygomaster_ids.json").read_bytes() == old_registry
    print("Determinismo e ID stabili OK")


## 3. Build → install → reinstall → uninstall

In [ ]:
with TemporaryDirectory() as tmp:
    project_root, ygomaster_dir = prepare_workspace(tmp)
    shop_path = ygomaster_dir / "Data" / "Shop.json"
    original_shop = shop_path.read_bytes()

    first = build_and_install_project(
        project_root, ygomaster_dir, ygoprodeck_client=FakeYgoProDeckClient(SOURCES)
    )
    assert len(first.installation.changed_files) == 3
    assert load_install_state(ygomaster_dir) is not None
    assert '"PerPackRarities": true' in shop_path.read_text(encoding="utf-8")
    assert first.installation.shop_file.read_bytes() == first.build.artifacts.shop.read_bytes()
    assert first.installation.shop_pack_odds_file.read_bytes() == first.build.artifacts.shop_pack_odds.read_bytes()

    second = build_and_install_project(
        project_root, ygomaster_dir, ygoprodeck_client=FakeYgoProDeckClient(SOURCES)
    )
    assert second.installation.changed_files == ()
    assert second.build.pack_ids == first.build.pack_ids

    uninstall_ygomaster_build(ygomaster_dir)
    assert shop_path.read_bytes() == original_shop
    assert not first.installation.shop_file.exists()
    assert not first.installation.shop_pack_odds_file.exists()
    assert load_install_state(ygomaster_dir) is None
    print("Installazione, reinstallazione, uninstall OK")


## 4. Manifest invalido: stop prima dei fetch e delle scritture

In [ ]:
with TemporaryDirectory() as tmp:
    project_root, ygomaster_dir = prepare_workspace(tmp)
    manifest_path = project_root / "config" / "project.yaml"
    manifest = yaml.safe_load(manifest_path.read_text(encoding="utf-8"))
    manifest["players"][0]["mini_packs"].pop()
    manifest_path.write_text(yaml.safe_dump(manifest, sort_keys=False), encoding="utf-8")

    client = FakeYgoProDeckClient(SOURCES)
    try:
        build_project(project_root, ygomaster_dir, ygoprodeck_client=client)
    except WorkflowError as exc:
        assert "Manifest validation failed" in str(exc)
    else:
        raise AssertionError("Il manifest invalido doveva fallire.")

    assert client.calls == []
    assert not (project_root / "build").exists()
    assert not (project_root / "state" / "ygomaster_ids.json").exists()
    print("Configurazione invalida bloccata OK")


## 5. ID YDK non risolto: registry e build invariati

In [ ]:
with TemporaryDirectory() as tmp:
    incomplete = dict(MAPPING)
    incomplete.pop(next(iter(incomplete)))
    project_root, ygomaster_dir = prepare_workspace(tmp, mapping=incomplete)
    registry_path = project_root / "state" / "ygomaster_ids.json"
    registry_path.parent.mkdir(parents=True)
    original = b'{"version": 1, "next_pack_id": 9000000, "pack_ids": {}}'
    registry_path.write_bytes(original)

    try:
        build_project(project_root, ygomaster_dir, ygoprodeck_client=FakeYgoProDeckClient(SOURCES))
    except WorkflowError as exc:
        assert "Domain validation failed" in str(exc)
        assert "unresolved-master-duel-id" in str(exc)
    else:
        raise AssertionError("Unresolved Master Duel ID doveva fallire.")

    assert registry_path.read_bytes() == original
    assert not (project_root / "build").exists()
    print("YDK mancante bloccato OK")


## 6. Fallimento scrittura finale: registry NON salvato

In [ ]:
with TemporaryDirectory() as tmp:
    project_root, ygomaster_dir = prepare_workspace(tmp)
    original_writer = workflow_module.write_ygomaster_build

    def simulated_failure(project, pack_ids, build_dir):
        if Path(build_dir) == project_root / "build":
            raise OSError("Simulated final write failure")
        return original_writer(project, pack_ids, build_dir)

    with patch.object(workflow_module, "write_ygomaster_build", side_effect=simulated_failure):
        try:
            build_project(project_root, ygomaster_dir, ygoprodeck_client=FakeYgoProDeckClient(SOURCES))
        except OSError as exc:
            assert "Simulated final write failure" in str(exc)
        else:
            raise AssertionError("La scrittura finale doveva fallire.")

    assert not (project_root / "state" / "ygomaster_ids.json").exists(), (
        "Correggi workflow.py: save_id_registry deve stare dopo la validazione finale."
    )
    assert list(project_root.glob(".ygomaster-packbuilder-*")) == []
    print("Errore finale lascia intatto il registry OK")


Se tutto passa, possiamo effettuare la **prima build live** con `build_project()` e un `YdkIds.txt` reale. È preferibile non installare ancora: prima verifichiamo che tutti e 13 i pack YGOPRODeck siano raggiungibili, abbiano la composizione corretta e che nessuna carta resti priva di Master Duel ID.